# Train on a free GPU (Colab / Kaggle)

Runs the same code as the local pipeline. On a T4 the training loop uses fp16 autocast automatically.
Checkpoints are written to Google Drive, and every run resumes from `last.pt` if the session is cut.

In [ ]:
# 1) Get the code: upload sidd_denoise.zip to the session (or clone your repo), then unpack it
import os, subprocess
if not os.path.exists("sidd_denoise"):
    subprocess.run("unzip -q sidd_denoise.zip -d sidd_denoise", shell=True, check=True)
%cd sidd_denoise
!pip -q install -e . pyiqa

In [ ]:
# 2) Persist runs/ and results/ on Drive (Colab only)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/sidd_denoise/runs", exist_ok=True)
    os.makedirs("/content/drive/MyDrive/sidd_denoise/results", exist_ok=True)
    for d in ["runs", "results"]:
        if not os.path.islink(d):
            subprocess.run(f"rm -rf {d} && ln -s /content/drive/MyDrive/sidd_denoise/{d} {d}", shell=True)
except ImportError:
    print("not on Colab: writing runs/ locally")

In [ ]:
# 3) Data (~7 GB) and pretrained weights. The York server is slow; parallel range requests help.
!python scripts/download_sidd.py --workers 16
!python scripts/download_weights.py
!python scripts/prepare_patches.py --workers 4

In [ ]:
# 4) Sanity check: tests plus a 200-iteration smoke run
!python -m pytest -q
!python scripts/train.py --config configs/smoke.yaml

In [ ]:
# 5) Full queue (resumable). On a T4 you can afford the untrimmed recipe, for example:
#    !python scripts/train.py --config configs/nafnet_w32_scratch.yaml --set sched.iters=40000 data.batch_size=16
!PY=python bash scripts/run_all.sh